# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jenilrupareliya5150-bit/FlyRankAi-ml-Track/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

## Research Question

### Which content pages should be reviewed first for refresh, expansion, protection, pruning, or monitoring?

This project uses observable historical search and traffic-performance signals from the FlyRank warehouse to build a leakage-safe machine-learning prioritization approach.

The goal is not to prove that changing a page will cause better search performance. Instead, the goal is to identify pages that are more likely to meet a defined future search-performance threshold and therefore deserve human review first.

The analysis uses historical signals such as Google Search Console impressions, clicks, search position, and available traffic/engagement measurements.

A Random Forest classifier will be evaluated against a simple baseline using a time-based validation design so that earlier observations are used to predict later observations.

The final output will be a ranked review queue with interpretable recommendations and reason codes.

In [12]:
import numpy as np
import pandas as pd



## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

## Dataset and Data Scope

This study uses the FlyRank pseudonymized warehouse release:

`flyrank_pseudonymized_warehouse_release_v20260703`

The primary modeling table is:

`fact_content_daily_performance`

This table contains daily observations at the client × content × date grain.

The full table contains **78,835,655 rows and 30 columns**. The available daily performance history spans:

**2025-01-27 to 2026-06-30**

The warehouse is processed using streaming/batched operations rather than loading the complete table into RAM. This allows full-dataset quality checks and feature processing while avoiding a memory-intensive full Pandas conversion.

The main signals available in the daily table include:

- Google Search Console impressions
- Google Search Console clicks
- Google Search Console search position
- Google Analytics 4 pageviews and sessions
- Organic, direct, referral, social and paid sessions
- AI-related traffic signals
- Scroll events

The dataset is an unbalanced panel, meaning different clients have different amounts of historical coverage. Availability flags are therefore treated as data-quality information rather than automatically interpreting missing measurements as zero performance.

Pseudonymized identifiers such as `client_hash_id` and `content_hash_id` are retained for defining observations and grouping, but are not used as predictive model features.

Future observations are not used as model inputs. They are used only when constructing the future-performance target, ensuring that the prediction task remains leakage-safe.

The final research output will not expose client names, domains, URLs, private queries, credentials, or other private identifying information.

In [7]:
# SECTION 2 — HUGGING FACE TOKEN + STREAMING DATASET

from google.colab import userdata
from datasets import load_dataset
import pandas as pd

# Read Hugging Face token from Colab Secrets
HF_TOKEN = userdata.get("HF_TOKEN")

if HF_TOKEN is None:
    raise ValueError(
        "HF_TOKEN Secret not found. "
        "Please add your Hugging Face token to Colab Secrets."
    )

print("✅ Hugging Face Token Loaded Successfully")


# Load the COMPLETE warehouse in streaming mode
# IMPORTANT: The complete 78M+ rows are NOT loaded into RAM.

dataset = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    split="train",
    streaming=True,
    token=HF_TOKEN
)

print("✅ Full warehouse connected in streaming mode")
print("Dataset type:", type(dataset))

✅ Hugging Face Token Loaded Successfully


README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

✅ Full warehouse connected in streaming mode
Dataset type: <class 'datasets.iterable_dataset.IterableDataset'>


In [8]:

# CHECK DATASET SCHEMA


print("Number of columns:", len(dataset.features))

print("\nColumns:")
for col in dataset.features:
    print("-", col)

print("\nDataset features:")
display(dataset.features)

Number of columns: 30

Columns:
- report_date
- client_hash_id
- content_hash_id
- client_has_gsc
- client_has_ga4
- gsc_data_available
- ga4_data_available
- gsc_impressions
- gsc_clicks
- gsc_sum_position
- gsc_avg_position
- ga4_pageviews
- ga4_sessions
- ga4_users
- ga4_engaged_sessions
- ga4_total_engagement_sec
- sessions_organic
- sessions_direct
- sessions_referral
- sessions_social
- sessions_paid
- sessions_ai
- ai_chatgpt
- ai_perplexity
- ai_gemini
- ai_copilot
- ai_claude
- ai_meta
- ai_other
- scroll_events

Dataset features:


{'report_date': Value('date32'),
 'client_hash_id': Value('string'),
 'content_hash_id': Value('string'),
 'client_has_gsc': Value('bool'),
 'client_has_ga4': Value('bool'),
 'gsc_data_available': Value('bool'),
 'ga4_data_available': Value('bool'),
 'gsc_impressions': Value('int64'),
 'gsc_clicks': Value('int64'),
 'gsc_sum_position': Value('int64'),
 'gsc_avg_position': Value('float64'),
 'ga4_pageviews': Value('int64'),
 'ga4_sessions': Value('int64'),
 'ga4_users': Value('int64'),
 'ga4_engaged_sessions': Value('int64'),
 'ga4_total_engagement_sec': Value('int64'),
 'sessions_organic': Value('int64'),
 'sessions_direct': Value('int64'),
 'sessions_referral': Value('int64'),
 'sessions_social': Value('int64'),
 'sessions_paid': Value('int64'),
 'sessions_ai': Value('int64'),
 'ai_chatgpt': Value('int64'),
 'ai_perplexity': Value('int64'),
 'ai_gemini': Value('int64'),
 'ai_copilot': Value('int64'),
 'ai_claude': Value('int64'),
 'ai_meta': Value('int64'),
 'ai_other': Value('int64')

In [9]:
sample_rows = list(dataset.take(5))

sample_df = pd.DataFrame(sample_rows)

print("Sample shape:", sample_df.shape)

display(sample_df)

Sample shape: (5, 30)


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115,...,0,0,0,0,0,0,0,0,0,0
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358,...,0,0,0,0,0,0,0,0,0,0
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34,...,0,0,0,0,0,0,0,0,0,0
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140,...,0,0,0,0,0,0,0,0,0,0
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89,...,0,0,0,0,0,0,0,0,0,0


In [ ]:

BATCH_SIZE = 50_000

# Running statistics
total_rows = 0
missing_counts = None

min_date = None
max_date = None

unique_clients = set()
unique_content = set()

# Numeric columns from the schema
numeric_columns = [
    col for col, feature in dataset.features.items()
    if str(feature).startswith(("Value('int", "Value('float"))
]

print("Numeric columns:", numeric_columns)
print("\nStarting full-dataset EDA...")
print("Batch size:", BATCH_SIZE)


# Process the complete warehouse batch-by-batch

for batch in dataset.iter(batch_size=BATCH_SIZE):

    batch_df = pd.DataFrame(batch)

    batch_rows = len(batch_df)
    total_rows += batch_rows


    # Missing values

    batch_missing = batch_df.isna().sum()

    if missing_counts is None:
        missing_counts = batch_missing
    else:
        missing_counts = missing_counts.add(
            batch_missing,
            fill_value=0
        )


    # Date range

    batch_dates = pd.to_datetime(
        batch_df["report_date"],
        errors="coerce"
    )

    batch_min = batch_dates.min()
    batch_max = batch_dates.max()

    if min_date is None or batch_min < min_date:
        min_date = batch_min

    if max_date is None or batch_max > max_date:
        max_date = batch_max

    # Unique clients/content
    unique_clients.update(
        batch_df["client_hash_id"].dropna().unique()
    )

    unique_content.update(
        batch_df["content_hash_id"].dropna().unique()
    )

# FINAL FULL-DATASET SUMMARY

print("\n" + "=" * 60)
print("FULL WAREHOUSE EDA SUMMARY")
print("=" * 60)

print("Total rows:", total_rows)
print("Total columns:", len(dataset.features))

print("\nDate range:")
print("Minimum date:", min_date)
print("Maximum date:", max_date)

print("\nUnique clients:", len(unique_clients))
print("Unique content pages:", len(unique_content))

# Missing-value summary

missing_summary = pd.DataFrame({
    "missing_count": missing_counts.astype(int),
    "missing_percentage": (
        missing_counts / total_rows * 100
    ).round(4)
})

missing_summary = missing_summary.sort_values(
    "missing_count",
    ascending=False
)

print("\nMissing-value summary:")
display(missing_summary)

Numeric columns: ['gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']

Starting full-dataset EDA...
Batch size: 50000


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

## 5. Limitations

*What this work cannot claim.*

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
